# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub pandas

import os, getpass
import duckdb
import pandas as pd

# Token: use Colab Secret named HF_TOKEN (preferred)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}
print("Connected to warehouse.")

Connected to warehouse.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content page (content_hash_id) on one day (report_date) for one client.  
For the refresh lane I will later aggregate to **one row per content page**.

**Time window for this contract:** I will develop on the mid-panel month **2026-03**  
I treat the final month (June 2026 / `_sample`) as a sealed test month and will not use it for label logic.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (knowable before decision):**  
- gsc_impressions (last 30 / prev 30)  
- gsc_clicks  
- gsc_avg_position  
- content age / days since last update (from dim_content)  
- engagement / sessions signals when ga4_data_available = TRUE  

**Label / proxy:**  
- is_declining (impressions dropped >20% from prev-30 to last-30 window)

**Context (grouping / join only):**  
- client_hash_id, content_hash_id  

**Excluded:**  
- trend_pct / trend_direction (label-derived → leakage)  
- Any product health_score or priority_score (not in the release and would be circular)  
- Raw URLs, titles, keywords (privacy + not needed)

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Query 1: Grain check (should return 0 rows) ---
print("=== 1. Grain check (one row = report_date × client × content) ===")
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(grain)
print("Rows returned:", len(grain), "(0 means grain is clean)\n")

# --- Query 2: Row count + date span for March 2026 ---
print("=== 2. Slice size and date span (month=2026-03) ===")
stats = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT content_hash_id) AS contents,
        COUNT(DISTINCT client_hash_id) AS clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print(stats)

# --- Query 3: Availability filter (ga4_data_available = TRUE) ---
print("\n=== 3. Availability – rows with ga4_data_available = TRUE ===")
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print(avail)

=== 1. Grain check (one row = report_date × client × content) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
Rows returned: 0 (0 means grain is clean)

=== 2. Slice size and date span (month=2026-03) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows  contents  clients   min_date   max_date
0  9841378    331437       55 2026-03-01 2026-03-31

=== 3. Availability – rows with ga4_data_available = TRUE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_true_rows
0     9841378       413966.0


## 4. Data limits

*What can this data never tell you?*

- History depth is unbalanced: many clients start later, so a global calendar window will treat “no tracking yet” as zero traffic.
- Early rows for some clients have GA4 columns zero-filled with `ga4_data_available = FALSE` — those zeros are not real “no engagement”.
- The final month (June 2026 / `_sample`) is the natural outcome window of any past→future label, so I must not develop label logic on it.
- This data cannot tell me *why* a page declined (algorithm change, seasonality, competitor, content quality) — only that the observed metrics moved.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.